# 2 · Filtering

Filtering removes interactions, users and items from the data WarpRec has read,
before it is split. This guide runs every filter on MovieLens-100K and shows
what each one keeps, with the size of the data before and after:

- where filtering runs, and the two ways to call it: the filter classes with
  `apply_filtering`, and the `filtering` section of a configuration;
- rating thresholds: `MinRating`, `UserAverage`, `ItemAverage`;
- activity bounds: `UserMin`, `UserMax`, `ItemMin`, `ItemMax`;
- k-cores: `IterativeKCore` and `NRoundsKCore`, and how they differ;
- per-user history length: `UserHeadN`, `UserTailN`;
- drop lists: `DropUser`, `DropItem`;
- why the order of the filters matters.

**Requirements:** `pip install warprec jupyter`. Runs in a few seconds; it
downloads MovieLens-100K (5 MB) into `guides/data/` if guide 1 has not.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

logger.disable("warprec")

`movielens_100k()` downloads the dataset and writes `ratings.tsv` with a header,
exactly as guide 1 does step by step. The reader returns the frame the filters
work on.

In [2]:
from warprec.data.reader import LocalReader

folder = movielens_100k()
data = LocalReader().read_tabular(local_path=str(folder / "ratings.tsv"), sep="\t")

Every table below compares the data before and after a filter by rows, distinct
users, distinct items and density (rows over users × items, in %).

In [3]:
def stats(frame) -> dict:
    """Size of an interaction frame."""
    rows, users, items = (
        len(frame),
        frame["user_id"].n_unique(),
        frame["item_id"].n_unique(),
    )
    return {
        "rows": rows,
        "users": users,
        "items": items,
        "density %": round(100 * rows / (users * items), 2),
    }


def compare(**frames) -> pd.DataFrame:
    """One row of stats per frame, starting from the unfiltered data."""
    rows = {"before": stats(data)} | {k: stats(f) for k, f in frames.items()}
    return pd.DataFrame.from_dict(rows, orient="index")


compare()

,rows,users,items,density %
before,100000,943,1682,6.3


## Where filtering runs

When a configuration reads a single file (`loading_strategy: dataset`),
`initialize_datasets` does, in this order:

1. read the file;
2. call the callback's `on_data_reading` hook on the raw frame (guide 16 covers
   callbacks);
3. apply the filters, in the order the configuration lists them;
4. split (guide 3).

So filters see whatever the callback returned, and the splitter sees only what
the filters kept. Data that is already split (`loading_strategy: split`) is
**not** filtered: the `filtering` section is ignored there, without a warning.

## Filters through the Python API

Each filter is a class in `warprec.data.filtering`, registered under its name in
`filter_registry`, which is how a configuration finds it. Names are looked up
case-insensitively.

In [4]:
from warprec.utils.registry import filter_registry

filter_registry.list_registered()

['MINRATING',
 'USERAVERAGE',
 'ITEMAVERAGE',
 'USERMIN',
 'USERMAX',
 'ITEMMIN',
 'ITEMMAX',
 'ITERATIVEKCORE',
 'NROUNDSKCORE',
 'USERHEADN',
 'USERTAILN',
 'DROPUSER',
 'DROPITEM']

A filter is built with its parameters and called on a frame. These two are the
same filter: one from its class, one from the registry by name.

In [5]:
from warprec.data.filtering import MinRating

by_class = MinRating(min_rating=4)
by_name = filter_registry.get("MinRating", min_rating=4)
len(by_class(data)) == len(by_name(data))

True

Filters find their columns through `user_id_label`, `item_id_label`,
`rating_label` and `timestamp_label`, which default to WarpRec's names. A
configuration passes the reader's `labels`, so renamed columns (guide 1) need
nothing more there.

`apply_filtering` runs a list of filters in order, logs the size after each one
and raises if one leaves nothing. With the log on:

In [6]:
from warprec.data.filtering import UserMin, apply_filtering

logger.enable("warprec")
filtered = apply_filtering(
    data, [MinRating(min_rating=4), UserMin(min_interactions=20)]
)
logger.disable("warprec")

08-10-2026 12:22:13 - WarpRec - 📝 Applying filters to the dataset. Initial dataset size: 100000 rows.


08-10-2026 12:22:13 - WarpRec - 📊 After filter 1/2 (MinRating): 55375 rows


08-10-2026 12:22:13 - WarpRec - 📊 After filter 2/2 (UserMin): 52244 rows


08-10-2026 12:22:13 - WarpRec - ✅ Filtering process completed. Final dataset size after filtering: 52244. Total filtering time: 0.00 seconds.


## Rating thresholds

- `MinRating(min_rating)` keeps the rows whose rating is **at least** `min_rating`.
- `UserAverage()` keeps the rows rated **above** that user's mean rating;
  `ItemAverage()` does the same against the item's mean. Both comparisons are
  strict, so a rating equal to the mean is removed too.

All three look only at the rating, but a user or item whose every row fails
the test disappears with them.

In [7]:
from warprec.data.filtering import ItemAverage, UserAverage

compare(
    **{
        "MinRating(4)": MinRating(min_rating=4)(data),
        "UserAverage()": UserAverage()(data),
        "ItemAverage()": ItemAverage()(data),
    }
)

,rows,users,items,density %
before,100000,943,1682,6.30
MinRating(4),55375,942,1447,4.06
UserAverage(),54194,943,1483,3.88
ItemAverage(),53301,943,1516,3.73


`MinRating(4)` keeps 55,375 of 100,000 rows and loses one user, whose ratings
are all below 4. `UserAverage` keeps every user (each one has some rating above
their own mean) but drops items nobody rated above their personal mean.

The `rating` column is read even with `rating_type: implicit`; the 1s are
stored when the dataset is built, after filtering. So a rating filter works on
an implicit configuration, and is a common way to turn ratings into positives
(the configuration at the end of this guide does that).

## Activity bounds

`UserMin(min_interactions)` keeps the users with at least that many rows,
`UserMax(max_interactions)` those with at most that many; `ItemMin` and
`ItemMax` do the same for items.
Each counts the rows of the frame it receives, once: `ItemMin` does not look
at users again, and vice versa.

In [8]:
from warprec.data.filtering import ItemMax, ItemMin, UserMax

compare(
    **{
        "UserMin(50)": UserMin(min_interactions=50)(data),
        "UserMax(50)": UserMax(max_interactions=50)(data),
        "ItemMin(50)": ItemMin(min_interactions=50)(data),
        "ItemMax(50)": ItemMax(max_interactions=50)(data),
    }
)

,rows,users,items,density %
before,100000,943,1682,6.30
UserMin(50),88471,568,1681,9.27
UserMax(50),11779,380,980,3.16
ItemMin(50),83715,943,603,14.72
ItemMax(50),16785,898,1089,1.72


`UserMin` and `ItemMin` raise the density; `UserMax` and `ItemMax` keep the
light users and the long tail, which is what a cold-start study filters for.

## k-cores

Both k-core filters alternate `UserMin(k)` and `ItemMin(k)`. Removing items can
push a user back under `k`, so one pass is not enough for a k-core.

- `NRoundsKCore(rounds, min_interactions)` runs at most `rounds` passes, and
  stops earlier if a pass removes nothing.
- `IterativeKCore(min_interactions)` runs passes until one removes nothing: the
  result is the k-core, where every user and item has at least `k` rows.

The last two columns count the users and items still under `k = 50`:

In [9]:
def under(frame, k: int) -> dict:
    """How many users and items have fewer than k rows in frame."""
    df = frame.to_native().to_pandas()
    return {
        "users < k": int((df.user_id.value_counts() < k).sum()),
        "items < k": int((df.item_id.value_counts() < k).sum()),
    }


from warprec.data.filtering import IterativeKCore, NRoundsKCore

k = 50
cores = {
    f"NRoundsKCore({r}, 50)": NRoundsKCore(rounds=r, min_interactions=k)(data)
    for r in range(1, 6)
}
cores["IterativeKCore(50)"] = IterativeKCore(min_interactions=k)(data)
pd.DataFrame.from_dict(
    {name: stats(f) | under(f, k) for name, f in cores.items()}, orient="index"
)

,rows,users,items,density %,users < k,items < k
"NRoundsKCore(1, 50)",72295,568,573,22.21,49,0
"NRoundsKCore(2, 50)",69612,519,562,23.87,4,0
"NRoundsKCore(3, 50)",69369,515,561,24.01,2,0
"NRoundsKCore(4, 50)",69222,513,560,24.10,0,0
"NRoundsKCore(5, 50)",69222,513,560,24.10,0,0
IterativeKCore(50),69222,513,560,24.10,0,0


After one round 49 users have fallen under 50 again. Round 4 is the last that
removes anything, so from 4 rounds on `NRoundsKCore` returns the same frame as
`IterativeKCore`, which ran a fifth pass to find that out. `NRoundsKCore` is the
cheaper, approximate choice when the k-core takes many passes on a large
dataset.

A threshold can leave nothing. The filter on its own returns an empty frame;
`apply_filtering` stops with an error naming the filter:

In [10]:
print(len(IterativeKCore(min_interactions=100)(data)), "rows")
try:
    apply_filtering(data, [IterativeKCore(min_interactions=100)])
except ValueError as error:
    print(error)

0 rows
Dataset is empty after applying filter 1/1: IterativeKCore. Please check the filtering criteria.


## First and last N per user

`UserHeadN(num_interactions)` keeps each user's first N rows by `timestamp`,
`UserTailN(num_interactions)` the last N. Rows with the same timestamp keep
their order in the file; without a `timestamp` column the file order is used.

In [11]:
from warprec.data.filtering import UserHeadN, UserTailN

head, tail = UserHeadN(num_interactions=20)(data), UserTailN(num_interactions=20)(data)
compare(**{"UserHeadN(20)": head, "UserTailN(20)": tail})

,rows,users,items,density %
before,100000,943,1682,6.30
UserHeadN(20),18860,943,1003,1.99
UserTailN(20),18860,943,1377,1.45


Every MovieLens-100K user has at least 20 ratings, so both keep 20 rows for
each of the 943 users. They differ in the items: the users' first 20 ratings
cover 1,003 items, their last 20 cover 1,377. The output is sorted by user and
timestamp:

In [12]:
head.to_native().head(3)

user_id,item_id,rating,timestamp
i64,i64,i64,i64
1,168,5,874965478
1,172,5,874965478
1,165,5,874965518


## Drop lists

`DropUser(user_ids_to_filter)` and `DropItem(item_ids_to_filter)` remove every
row of the given ids, a single id or a list. The ids are the ones in the file,
not WarpRec's indices. Here the two most active users and the most rated item:

In [13]:
from warprec.data.filtering import DropItem, DropUser

counts = data.to_native().to_pandas()
top_users = counts.user_id.value_counts().index[:2].tolist()
top_item = int(counts.item_id.value_counts().index[0])
titles = pd.read_csv(folder / "items.tsv", sep="\t", index_col="item_id").title
print("users", top_users, "| item", top_item, titles[top_item])

compare(
    **{
        f"DropUser({top_users})": DropUser(user_ids_to_filter=top_users)(data),
        f"DropItem({top_item})": DropItem(item_ids_to_filter=top_item)(data),
    }
)

users [405, 655] | item 50 Star Wars (1977)


,rows,users,items,density %
before,100000,943,1682,6.30
"DropUser([405, 655])",98578,941,1634,6.41
DropItem(50),99417,943,1681,6.27


Dropping two users also removes 48 items that only they had rated.

## Order matters

Filters do not commute. `MinRating(4)` then `UserMin(20)` keeps the users with
at least 20 ratings of 4 or 5. The other order counts ratings of any value; every
MovieLens-100K user has at least 20, so `UserMin` removes nothing and the result
is `MinRating(4)` alone, with 239 users left under 20 rows.

In [14]:
rating_then_users = apply_filtering(
    data, [MinRating(min_rating=4), UserMin(min_interactions=20)]
)
users_then_rating = apply_filtering(
    data, [UserMin(min_interactions=20), MinRating(min_rating=4)]
)
orders = {
    "before": data,
    "MinRating → UserMin": rating_then_users,
    "UserMin → MinRating": users_then_rating,
}
pd.DataFrame.from_dict(
    {
        name: stats(f) | {"users < 20": under(f, 20)["users < k"]}
        for name, f in orders.items()
    },
    orient="index",
)

,rows,users,items,density %,users < 20
before,100000,943,1682,6.30,0
MinRating → UserMin,52244,703,1439,5.16,0
UserMin → MinRating,55375,942,1447,4.06,239


## The same thing from a configuration

The `filtering` section maps filter names to their parameters (`{}` for none).
The filters run top to bottom, in the order they are written. This
configuration keeps ratings of 4 and 5 and reads them as implicit positives,
then takes the 10-core:

In [15]:
print(Path("configs/filtering.yml").read_text())

# Keep ratings of 4 and 5 as positives, then the 10-core of what is left.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  sep: "\t"
  header: true
  rating_type: implicit
filtering:
  MinRating:
    min_rating: 4
  IterativeKCore:
    min_interactions: 10
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



The configuration is checked when it is loaded: an unknown filter name, or a
filter that cannot be built from its parameters (a missing one, an invalid
value), raises there. A key the filter does not take is ignored silently, so a
stray key next to the right ones goes unnoticed.

The callback below prints the size of the frame it is handed, to show that
`on_data_reading` runs before the filters. With the log on:

In [16]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration


class ShowSize(WarpRecCallback):
    def on_data_reading(self, data):
        print(f"on_data_reading: {len(data)} rows")
        return data


logger.enable("warprec")
config = load_design_configuration("configs/filtering.yml")
main, _, _ = initialize_datasets(ReaderFactory.get_reader(config), ShowSize(), config)
logger.disable("warprec")

08-10-2026 12:22:13 - WarpRec - 📝 Reading design configuration file in: configs/filtering.yml


08-10-2026 12:22:13 - WarpRec - 📝 Reading process completed correctly.


on_data_reading: 100000 rows
08-10-2026 12:22:13 - WarpRec - 📝 Applying filters to the dataset. Initial dataset size: 100000 rows.


08-10-2026 12:22:13 - WarpRec - 📊 After filter 1/2 (MinRating): 55375 rows


08-10-2026 12:22:13 - WarpRec - 📊 After filter 2/2 (IterativeKCore): 52764 rows


...

08-10-2026 12:22:13 - WarpRec - 📊 Number of users: 887      Number of items: 822      Transactions: 47088


08-10-2026 12:22:13 - WarpRec - 📝 -----------------------------------------------------------------------


08-10-2026 12:22:13 - WarpRec - 📝 -------------------------------Test set-------------------------------


08-10-2026 12:22:13 - WarpRec - 📊 Number of users: 887      Number of items: 797      Transactions: 5676


08-10-2026 12:22:13 - WarpRec - 📝 ----------------------------------------------------------------------


The callback gets all 100,000 rows; the log then reports the size after each
filter, before the split. The training matrix holds only 1s:

In [17]:
print(main.get_dims(), "users x items |", set(main.train_set.get_sparse().data))

(887, 822) users x items | {np.float32(1.0)}


`config.get_filters()` returns the filter objects the configuration describes,
in the order `initialize_datasets` applies them. The two configurations below
differ only in the order of their filters:

In [18]:
print(Path("configs/users-then-rating.yml").read_text())

# Keep users with at least 20 ratings, then only their ratings of 4 and 5.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  sep: "\t"
  header: true
  rating_type: explicit
filtering:
  UserMin:
    min_interactions: 20
  MinRating:
    min_rating: 4
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [19]:
rows = {}
for name in ["rating-then-users", "users-then-rating"]:
    config = load_design_configuration(f"configs/{name}.yml")
    main, _, _ = initialize_datasets(
        ReaderFactory.get_reader(config), WarpRecCallback(), config
    )
    order = " → ".join(type(f).__name__ for f in config.get_filters())
    rows[name] = {
        "filters": order,
        "rows kept": len(apply_filtering(data, config.get_filters())),
        "train users x items": main.get_dims(),
    }
pd.DataFrame.from_dict(rows, orient="index")

,filters,rows kept,train users x items
rating-then-users,MinRating → UserMin,52244,"(703, 1374)"
users-then-rating,UserMin → MinRating,55375,"(942, 1383)"


The filters each configuration builds keep the same rows as the two API runs
above, and the datasets differ accordingly. The dimensions are those of the
training split, which is what defines the catalogue (guide 1), so they count
fewer items than the filtered frames.

The section is a YAML mapping, so a filter name can appear only once. Written
twice, YAML keeps the last parameters at the first position, without an error;
a filter that is needed twice has to be applied through the API.

## Summary of the `filtering` section

| key | parameters | keeps |
|---|---|---|
| `MinRating` | `min_rating` | rows with rating ≥ `min_rating` |
| `UserAverage`, `ItemAverage` | none (`{}`) | rows rated above the user's / item's mean |
| `UserMin`, `UserMax` | `min_interactions` / `max_interactions` | users with ≥ / ≤ that many rows |
| `ItemMin`, `ItemMax` | `min_interactions` / `max_interactions` | items with ≥ / ≤ that many rows |
| `IterativeKCore` | `min_interactions` | the k-core: user and item passes until nothing changes |
| `NRoundsKCore` | `rounds`, `min_interactions` | at most `rounds` user and item passes |
| `UserHeadN`, `UserTailN` | `num_interactions` | each user's first / last N rows by timestamp |
| `DropUser`, `DropItem` | `user_ids_to_filter` / `item_ids_to_filter` | everything but those ids |

Filters run in the order written, after `on_data_reading` and before the
split, and only when a single file is read. See the
[Filtering configuration](https://warprec.readthedocs.io/en/latest/configuration/filtering/)
page; guide 3 continues with splitting.